# LoRA fine-tuning of Qwen3-1.7B for cited answers

Trains a LoRA adapter on `train_chat.jsonl` (315 examples, papers P003 P004 P005 P008),
selects the epoch with the lowest loss on `val_chat.jsonl` (56 examples, paper P007), and
exports a Q4_K_M GGUF for Ollama, the same quantization as the `qwen3:1.7b` baseline.

**Run on Kaggle:** Settings: Accelerator = GPU T4, Internet = on. Add a private dataset
containing `train_chat.jsonl` and `val_chat.jsonl` from `data/train/build/`. Run all.
Repeat with `SEED = 1` and `SEED = 2` (one run each, about 30 min).
**Colab:** Runtime > Change runtime type > T4 GPU, upload the two files to `/content`,
Run all, and allow Drive access when asked; outputs are saved to
`MyDrive/research-assistant/<run>`.

The data contains no P001 (private), P002 or P006 (test) text; section 2 checks this.

In [ ]:
!pip install -q unsloth

## 1. Configuration
Change only `SEED` between runs. Each value has a reason:
- **LoRA r=16, alpha=16, all linear layers:** a common setting for small SFT sets. With 315
  examples, a larger rank mostly adds capacity to memorize.
- **16-bit LoRA, not QLoRA:** 1.7B parameters fit in 16-bit on a T4 (about 3.4 GB), so
  4-bit quantization during training would add error for no memory benefit.
- **lr 2e-4, cosine, 5% warmup, 3 epochs, keep the best epoch by validation loss:**
  standard LoRA values. Small data overfits fast, so the epoch is chosen on P007.
- **Effective batch 8** (2 x 4 accumulation): about 40 optimizer steps per epoch.
- **Loss on the answer only:** about 90% of each example is retrieved passages. Training
  on them would teach the model to memorize paper text, the opposite of the goal.

In [ ]:
SEED = 0                      # run 0, 1, 2
BASE_MODEL = "unsloth/Qwen3-1.7B"
MAX_LEN = 4096                # longest example is 3,959 tokens
LORA_R, LORA_ALPHA = 16, 16
LR, EPOCHS, BATCH, GRAD_ACC, WARMUP = 2e-4, 3, 2, 4, 0.05
GIT_COMMIT = ""               # paste `git rev-parse --short HEAD` for the record
RUN_NAME = f"ra-qwen3-1.7b-s{SEED}"

### Colab only: connect Google Drive now
Colab deletes its files when the session ends, so the outputs are copied to Drive at the
end. Drive asks for permission here, at the start, so a *Run all* does not stop and wait
for you after training. On Kaggle this cell does nothing.

In [ ]:
import importlib.util
ON_COLAB = importlib.util.find_spec("google.colab") is not None
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = f"/content/drive/MyDrive/research-assistant/{RUN_NAME}"
    print("outputs will be copied to", DRIVE_DIR)

## 2. Load and check the data

In [ ]:
import glob, hashlib, json, re
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/content/{name}")
    assert hits, f"{name} not found: add it as a Kaggle dataset or upload it to /content"
    return hits[0]
paths = {s: find(f"{s}_chat.jsonl") for s in ("train", "val")}
data = {s: [json.loads(l) for l in open(p, encoding="utf-8")] for s, p in paths.items()}
sha = {s: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16] for s, p in paths.items()}

# guard: no private (P001) or test-paper (P002, P006) passages may be in training data
forbidden = re.compile(r"\[P00[126]_c\d+\]")
bad = [r["item_id"] for s in data for r in data[s] if forbidden.search(r["text"])]
assert not bad, f"forbidden papers in data: {bad[:5]}"
assert all(r["text"].count("<|im_start|>assistant\n<think>\n</think>\n") == 1
           for s in data for r in data[s]), "text is not in Ollama's qwen3 format"
print({s: len(v) for s, v in data.items()}, sha)

## 3. Model and LoRA adapter

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
torch.manual_seed(SEED)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=False, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)

lengths = [len(tokenizer(r["text"], add_special_tokens=False).input_ids) for s in data for r in data[s]]
print("tokens per example: max", max(lengths), "mean", sum(lengths) // len(lengths))
assert max(lengths) <= MAX_LEN, "an example would be truncated: raise MAX_LEN"

## 4. Trainer (loss on answers only)

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

ds = {s: Dataset.from_list([{"text": r["text"]} for r in data[s]]) for s in data}
# TRL renamed max_seq_length to max_length; its default (1024) would silently truncate
len_key = "max_length" if "max_length" in SFTConfig.__dataclass_fields__ else "max_seq_length"
cfg = SFTConfig(
    output_dir=f"out/{RUN_NAME}", dataset_text_field="text", packing=False, **{len_key: MAX_LEN},
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, num_train_epochs=EPOCHS, learning_rate=LR,
    lr_scheduler_type="cosine", warmup_ratio=WARMUP, weight_decay=0.0,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=EPOCHS,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=5, seed=SEED, data_seed=SEED, report_to="none")
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds["train"],
                     eval_dataset=ds["val"], args=cfg)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

# sanity: the tokens that carry loss must be the answer only, not the passages
ex = trainer.train_dataset[0]
kept = tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])
print("loss is computed on:", repr(kept[:300]))
assert "Passages:" not in kept and kept.count("[Section:") == 0, "loss mask covers the prompt"

## 5. Train

In [ ]:
before = trainer.evaluate()["eval_loss"]
print("validation loss before training:", round(before, 4))
trainer.train()
evals = [h for h in trainer.state.log_history if "eval_loss" in h]
for h in evals:
    print(f"epoch {h['epoch']:.0f}  eval_loss {h['eval_loss']:.4f}")
best = min(evals, key=lambda h: h["eval_loss"])
print("kept epoch", round(best["epoch"]), "eval_loss", round(best["eval_loss"], 4))

## 6. Quick look at three validation answers
Not an evaluation (that is `src/eval/eval_generation.py`, run locally through Ollama).
Only a check that the model answers in the cited format and stops at `<|im_end|>`.

In [ ]:
FastLanguageModel.for_inference(model)
cite = re.compile(r"\[P\d+_c\d+(?:,\s*P\d+_c\d+)*\]")
marker = "<|im_start|>assistant\n<think>\n</think>\n"
for r in data["val"][:3]:
    prompt = r["text"].split(marker)[0] + marker
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    answer = tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    print(r["item_id"], "| cites:", bool(cite.search(answer)), "|", answer[:300], "\n")

## 7. Save adapter, GGUF (Q4_K_M) and run record

In [ ]:
import os, platform, shutil
import transformers, trl, unsloth
model.save_pretrained(f"{RUN_NAME}-lora"); tokenizer.save_pretrained(f"{RUN_NAME}-lora")
model.save_pretrained_gguf(f"{RUN_NAME}-gguf", tokenizer, quantization_method="q4_k_m")
gguf = sorted(glob.glob(f"{RUN_NAME}-gguf*/**/*.gguf", recursive=True)
              + glob.glob(f"{RUN_NAME}-gguf*.gguf"), key=os.path.getsize)[-1]
shutil.copy(gguf, f"{RUN_NAME}.Q4_K_M.gguf")

record = {"run": RUN_NAME, "seed": SEED, "git_commit": GIT_COMMIT, "base_model": BASE_MODEL,
          "data_sha256_16": sha, "n": {s: len(v) for s, v in data.items()},
          "hparams": {"lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lr": LR, "epochs": EPOCHS,
                      "batch": BATCH, "grad_acc": GRAD_ACC, "warmup": WARMUP, "max_len": MAX_LEN},
          "val_loss_before": before, "best_epoch": best["epoch"], "best_val_loss": best["eval_loss"],
          "log_history": trainer.state.log_history,
          "versions": {"unsloth": unsloth.__version__, "transformers": transformers.__version__,
                       "trl": trl.__version__, "torch": torch.__version__,
                       "gpu": torch.cuda.get_device_name(0), "python": platform.python_version()}}
json.dump(record, open(f"{RUN_NAME}.json", "w"), indent=2)
shutil.make_archive(f"{RUN_NAME}-lora", "zip", f"{RUN_NAME}-lora")
print("download:", f"{RUN_NAME}.Q4_K_M.gguf", f"{RUN_NAME}.json", f"{RUN_NAME}-lora.zip")

## 8. Colab only: copy outputs to Drive
Checks each copy by size. On Kaggle, download the files from the Output panel instead.

In [ ]:
import os, shutil
if ON_COLAB:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for f in (f"{RUN_NAME}.Q4_K_M.gguf", f"{RUN_NAME}.json", f"{RUN_NAME}-lora.zip"):
        dest = shutil.copy(f, DRIVE_DIR)
        assert os.path.getsize(dest) == os.path.getsize(f), f"incomplete copy: {dest}"
        print(f"saved {dest} ({os.path.getsize(dest) / 1e6:.0f} MB)")
else:
    print("not on Colab: download the files from the Output panel")